In [2]:
from pathlib import Path

DATA_PATH = Path("../data/redwine_quality.csv")

DATA_PATH.exists()

True

In [3]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [5]:
df.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [6]:
X = df.drop(columns=["quality"])     # 피처 11개
y = df["quality"]                   # 타깃 (3~8 점수)

print(X.shape, y.shape)
print(y.value_counts().sort_index()) # 클래스 분포 확인

(1599, 11) (1599,)
quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [8]:
models = {
    # KNN: 거리 기반 → 스케일링 필수 (00~02)
    "KNN": Pipeline([
        ("sc", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]),
    # GaussianNB: 연속 측정값 → 정규분포 가정에 적합 (04)
    "GaussianNB": Pipeline([
        ("sc", StandardScaler()),
        ("nb", GaussianNB()),
    ]),
    # DecisionTree: 스케일링 불필요 (05)
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    # RandomForest: 스케일링 불필요, 앙상블 (06)
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
}

In [9]:
# 불균형이 심하니 accuracy와 f1_macro 둘 다 (소수 클래스도 반영)
for name, model in models.items():
    acc = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    f1 = cross_val_score(model, X, y, cv=5, scoring="f1_macro")
    print(f"{name:14s} | acc {acc.mean():.4f} | f1_macro {f1.mean():.4f}")

KNN            | acc 0.5203 | f1_macro 0.2589
GaussianNB     | acc 0.5235 | f1_macro 0.2960
DecisionTree   | acc 0.4840 | f1_macro 0.2777
RandomForest   | acc 0.5641 | f1_macro 0.2842


In [10]:
# quality 7 이상이면 좋은와인(1) 아니면 0
y_bin = (df["quality"] >= 7).astype(int)

print(y_bin.value_counts())
print("좋은 와인 비율:", y_bin.mean().round(3))

quality
0    1382
1     217
Name: count, dtype: int64
좋은 와인 비율: 0.136


In [11]:
for name, model in models.items():
    acc = cross_val_score(model, X, y_bin, cv=5, scoring="accuracy")
    f1 = cross_val_score(model, X, y_bin, cv=5, scoring="f1")
    print(f"{name:14s} | acc {acc.mean():.4f} | f1 {f1.mean():.4f}")

KNN            | acc 0.8380 | f1 0.3753
GaussianNB     | acc 0.8117 | f1 0.4976
DecisionTree   | acc 0.8130 | f1 0.3961
RandomForest   | acc 0.8618 | f1 0.3647


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y_bin, test_size=0.2, stratify=y_bin, random_state=42
)

final = models["GaussianNB"]
final.fit(X_train, y_train)
pred = final.predict(X_test)

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["나쁨", "좋음"]))

[[242  35]
 [ 14  29]]
              precision    recall  f1-score   support

          나쁨       0.95      0.87      0.91       277
          좋음       0.45      0.67      0.54        43

    accuracy                           0.85       320
   macro avg       0.70      0.77      0.73       320
weighted avg       0.88      0.85      0.86       320



In [13]:
# 앞서 cv=5 비교를 cv=10으로 (fold를 늘리면 순위가 달라지나?)
for name, model in models.items():
    acc = cross_val_score(model, X, y_bin, cv=10, scoring="accuracy")
    f1 = cross_val_score(model, X, y_bin, cv=10, scoring="f1")
    print(f"{name:14s} | acc {acc.mean():.4f} ± {acc.std():.4f} "
          f"| f1 {f1.mean():.4f} ± {f1.std():.4f}")

KNN            | acc 0.8449 ± 0.0624 | f1 0.3699 ± 0.1355
GaussianNB     | acc 0.8261 ± 0.1188 | f1 0.5487 ± 0.1591
DecisionTree   | acc 0.8286 ± 0.0654 | f1 0.4355 ± 0.1226
RandomForest   | acc 0.8749 ± 0.0415 | f1 0.4273 ± 0.1035


## 결론: 4개 모델 종합 비교 — 좋은 와인 판별

지금까지 배운 KNN·NaiveBayes·DecisionTree·RandomForest를 한 데이터에서 경쟁시켰다.

### 각 모델과 전처리 (모델 성격에 맞춤)
| 모델 | 전처리 | 이유 |
|---|---|---|
| KNN | StandardScaler | 거리 기반 → 스케일 민감 (00~02) |
| GaussianNB | StandardScaler | 연속 측정값, 정규분포 가정 (04) |
| DecisionTree | 없음 | 기준값 비교라 스케일 무관 (05) |
| RandomForest | 없음 | 트리 앙상블 (06) |

### 문제 정의의 진화
- **다중분류(quality 3~8)**: f1_macro 0.28 — 소수 클래스(3,8은 10·18개) 학습 불가로 폭락.
- **이진화(quality≥7 = 좋은 와인)**: f1 개선. "좋은 와인 판별"로 문제가 명확해짐.

### 핵심 발견: accuracy 1등 ≠ f1 1등 ⭐
| 모델 | accuracy | f1(좋은와인) |
|---|---|---|
| RandomForest | **0.86** (1등) | 0.43 |
| GaussianNB | 0.83 | **0.55** (1등) |

- RandomForest는 다수 클래스(나쁜 와인)에 안전빵 → acc 높지만 좋은 와인을 놓침.
- GaussianNB는 좋은 와인을 적극 발굴(recall 0.67) → f1 1등.
- **우리 목표는 "좋은 와인 찾기" → f1이 진짜 지표 → GaussianNB 선정.**
  (accuracy만 봤다면 잘못된 모델을 골랐을 것 — 02·03의 교훈)

### 왜 GaussianNB가 이겼나
피처(화학 성분)가 대체로 정규분포에 가까워, 정규분포를 가정하는 GaussianNB와 잘 맞음.
→ 04에서 배운 "데이터 성격이 모델을 결정한다"의 실전 증명.
(만능처럼 보이는 RandomForest를, 데이터에 맞는 GaussianNB가 이김)

### 검증
- 최종 GaussianNB: test f1 0.54, CV f1 0.50~0.55와 일치 → 과적합 없음, 선택이 옳음.
- cv=5, cv=10 모두 GaussianNB f1 1등 → 견고한 결론.
- 단, 표준편차 큼(±0.16): 좋은 와인이 13.6%로 드물어 fold마다 출렁임.

### 현실적 한계 (솔직하게)
- wine quality는 화학 성분으로 사람의 맛 평가를 맞히는 원래 어려운 문제.
- 좋은 와인 f1 0.54는 완벽하진 않으나, 13.6% 불균형 + 어려운 문제임을 감안하면 현실적 성능.
- 좋은 와인의 recall 0.67 — 실제 좋은 와인의 2/3를 발굴하므로 목적엔 부합.


In [14]:
from xgboost import XGBClassifier

# XGBoost: 부스팅 (순차적으로 오차 보완). 스케일링 불필요 (트리 기반)
models["XGBoost"] = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="logloss",
)

In [16]:
for name, model in models.items():
    acc = cross_val_score(model, X, y_bin, cv=5, scoring="accuracy")
    f1 = cross_val_score(model, X, y_bin, cv=5, scoring="f1")
    print(f"{name:14s} | acc {acc.mean():.4f} | f1 {f1.mean():.4f}")

KNN            | acc 0.8380 | f1 0.3753
GaussianNB     | acc 0.8117 | f1 0.4976
DecisionTree   | acc 0.8130 | f1 0.3961
RandomForest   | acc 0.8618 | f1 0.3647
XGBoost        | acc 0.8505 | f1 0.3975


In [17]:
from xgboost import XGBClassifier

# 불균형 대응: scale_pos_weight로 좋은 와인에 가중치
n_neg = (y_bin == 0).sum()
n_pos = (y_bin == 1).sum()
spw = n_neg / n_pos
print(f"scale_pos_weight = {spw:.2f}")

xgb_balanced = XGBClassifier(
    n_estimators=100, scale_pos_weight=spw,
    eval_metric="logloss", random_state=42,
)

acc = cross_val_score(xgb_balanced, X, y_bin, cv=5, scoring="accuracy")
f1 = cross_val_score(xgb_balanced, X, y_bin, cv=5, scoring="f1")
print(f"XGBoost(balanced) | acc {acc.mean():.4f} | f1 {f1.mean():.4f}")


scale_pos_weight = 6.37
XGBoost(balanced) | acc 0.8424 | f1 0.4320


In [19]:
# imblearn 필요 (uv add imbalanced-learn)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# SMOTE를 Pipeline에 넣으면 → CV의 각 fold train에만 적용 (누수 없음!)
xgb_smote = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("xgb", XGBClassifier(n_estimators=100, eval_metric="logloss", random_state=42)),
])
f1 = cross_val_score(xgb_smote, X, y_bin, cv=5, scoring="f1")
print(f"XGBoost+SMOTE(정직) | f1 {f1.mean():.4f}")


XGBoost+SMOTE(정직) | f1 0.4426


In [20]:
# 그들의 지표(f1_weighted)로 재비교 — 순위가 뒤집히나?
for name in ["GaussianNB", "RandomForest", "XGBoost"]:
    model = models[name]
    fw = cross_val_score(model, X, y_bin, cv=5, scoring="f1_weighted")
    fb = cross_val_score(model, X, y_bin, cv=5, scoring="f1")   # 좋은와인 기준
    print(f"{name:14s} | f1_weighted {fw.mean():.4f} | f1(좋은와인) {fb.mean():.4f}")


GaussianNB     | f1_weighted 0.8268 | f1(좋은와인) 0.4976
RandomForest   | f1_weighted 0.8458 | f1(좋은와인) 0.3647
XGBoost        | f1_weighted 0.8426 | f1(좋은와인) 0.3975


## 결론: 5개 모델 종합 비교 — "1등"은 지표가 정한다

KNN·NaiveBayes·DecisionTree·RandomForest·XGBoost를 한 데이터에서 경쟁시켰다.

### 모델별 전처리 (성격에 맞춤)
| 모델 | 전처리 | 이유 |
|---|---|---|
| KNN | StandardScaler | 거리 기반 (00~02) |
| GaussianNB | StandardScaler | 연속 측정값, 정규분포 가정 (04) |
| DecisionTree / RandomForest / XGBoost | 없음 | 트리 기반, 스케일 무관 (05·06) |

### 문제 정의의 진화
- 다중분류(quality 3~8): f1_macro 0.28 — 소수 클래스 학습 불가로 폭락.
- 이진화(quality≥7 = 좋은 와인, 13.6%): "좋은 와인 발굴"로 문제가 명확해짐.

### 핵심 발견 ①: 지표에 따라 1등이 뒤바뀐다 ⭐
| 모델 | f1_weighted | f1(좋은와인) |
|---|---|---|
| RandomForest | **0.8458** (1등) | 0.3647 |
| XGBoost | 0.8426 | 0.3975 |
| GaussianNB | 0.8268 | **0.4976** (1등) |

- f1_weighted(=accuracy 착시, 다수 클래스 지배) → RF/XGBoost 승.
- f1(좋은와인, 소수 클래스) → GaussianNB 승.
- **우리 목표는 "좋은 와인 발굴" → 좋은와인 f1이 맞는 지표 → GaussianNB 선정.**

### 핵심 발견 ②: 외부 "XGBoost 0.94"의 정체
외부 소스(Kaggle/GitHub)를 직접 확인한 결과:
- 0.94는 **accuracy 또는 f1_weighted** (불균형 착시). 우리도 accuracy는 0.85.
- **SMOTE를 썼으나 분할 전/후 미명시** → 분할 전이면 데이터 누수.
- 우리가 SMOTE를 정직하게(분할 후 train만, imblearn Pipeline) 적용 → f1 0.44에 그침.
→ 격차는 XGBoost의 실력이 아니라 **지표·측정 방식의 차이**.

### 검증
- 최종 GaussianNB: test f1 0.54, CV(cv=5,10) f1 0.50~0.55와 일치 → 과적합 없음.
- XGBoost에 튜닝(scale_pos_weight)·SMOTE까지 공정하게 줘도 좋은와인 f1은 GaussianNB 미달.

### 왜 GaussianNB인가
- 피처(화학 성분)가 정규분포에 가까움 → 정규분포 가정과 일치 (04의 실전 증명).
- 데이터가 작음(1599개) → 복잡한 부스팅이 이점을 못 살림.

### 현실적 한계 (솔직하게)
- wine quality는 화학 성분으로 사람의 맛 평가를 맞히는 원래 어려운 문제.
- 좋은 와인 f1 0.54는 완벽하진 않으나, 13.6% 불균형 감안 시 현실적. recall 0.67로 2/3 발굴.

### 오늘의 교훈
"누가 1등이냐"보다 **"무슨 목표로, 무슨 지표로 쟀나"**를 먼저 물어야 한다.
남의 높은 점수는 지표·누수·문제 정의를 확인하기 전엔 믿지 않는다.
